In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from constrerl.erl_schema import (
    entity_labels,
    relations,
    
)
from constrerl.annotator import Annotator, AnnotationTypes
from constrerl.annotation_model import (
    AnnotatedArticle,
    load_collection,
    
)

In [ ]:
dev_articles=load_collection("Dev")
train_articles=load_collection("Train")

In [ ]:
dev_articles

In [ ]:
test_article=dev_articles[list(dev_articles.keys())[0]]
test_article

In [ ]:
test_article.metadata.abstract

In [ ]:
from constrerl.annotator import extract_noun_phrases
print(extract_noun_phrases(test_article.metadata.abstract), test_article.metadata.abstract)

In [ ]:
# model = Llama.from_pretrained(
#     "NousResearch/Hermes-3-Llama-3.2-3B-GGUF",
#     filename="*.Q8_0.gguf",
#     n_gpu_layers=-1,
#     n_ctx=8096,
#     temperature=0.1,
# )

In [ ]:
from constrerl.beam_search.grammar import GBNF_PARSER, test_against_grammar

In [ ]:
test_grammar = """
root ::= ent-list
entity ::= entity-type" ("entity-str")"
arbitrary-str ::= (([0-9a-fA-F]|" "){1, 4})
entity-type ::= "Anatomical Location"|"Animal"|"Biomedical Technique"|"Bacteria"|"Chemical"|"Dietary Supplement"|"DDF"|"Drug"|"Food"|"Gene"|"Human"|"Microbiome"|"Statistical Technique"
entity-str ::= "Orthopedic Surgery"|"Gut Microbiome Dysbiosis"|"Intestinal Barrier Dysfunction"|"Prodromal Alzheimer Disease Patients"|"Prospective Observational Cohort Study"
ent-list ::= entity ("\\n" entity)*
"""
test_grammar_parsed = GBNF_PARSER.parse(test_grammar)
print(test_grammar_parsed.pretty())

In [ ]:
test_against_grammar("DDF", test_grammar_parsed)


In [ ]:
from llama_cpp import Llama

# model_path = "quants/hermes-3-2-3B-lora-entities.gguf"
model_path = "quants/hermes-3-2-3B.gguf"
model = Llama(
    model_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    logits_all=True,
    verbose=False
    # draft_model=LlamaPromptLookupDecoding(num_pred_tokens=10),
)

In [ ]:
entities=[lbl["label"] for lbl in entity_labels]
entities

In [ ]:
test_articles = {"test": test_article.metadata}

In [ ]:
# annotator_train = AnnotatorHelper(
#     gen_tokens=512,
#     top_k=5,
#     add_rag=True
# )
# annotator_train.model = model
# annotator_train.load_articles(train_articles)
# annotator_train.save_articles(Path("./data/annotations/prepared_train.json"))

In [ ]:
dev_articles_reduced = {k: v for i, (k, v) in enumerate(dev_articles.items()) if i < 1}
# dev_articles_reduced = dev_articles #{k: v for i, (k, v) in enumerate(dev_articles.items()) if i < 1}

In [ ]:
from constrerl.annotator import AnnotatorHelper, BeamSearchConfig
from pathlib import Path


annotator = AnnotatorHelper(
    gen_tokens=4,
    top_k=5,
    naive_annotations=True,
    naive_only=False,
    # add_rag=True,
    beam_search=BeamSearchConfig(top_k=2, max_depth=3),
)
annotator.model = model
# annotator.load_articles(dev_articles | train_articles)
# annotator.save_articles(Path("./data/annotations/prepared_dev_train.json"))

annotator.load_articles_from_path(Path("./data/annotations/prepared_dev.json"))
annotator.load_concepts(Path("./data/Annotations/uri_collection_concepts.json"))

In [ ]:
annotated_articles = annotator.annotate(
    {
        id: article.metadata for id, article in dev_articles_reduced.items()
    },  
    annotate=[AnnotationTypes.ENTITY]
)

In [ ]:
annotator.model._scores

In [ ]:
annotated_articles

In [ ]:
annotated_articles[list(annotated_articles.keys())[0]].entities

In [ ]:
# annotator.add_concept_uris(annotated_articles)

In [ ]:
from constrerl.utils import prepare_for_eval

In [ ]:
from constrerl.eval.evaluate import (
    eval_submission_mention_level_RE,
    eval_submission_NER,
    eval_submission_NERD,
    eval_submission_concept_level_RE,
)


def tuple_to_scores(tup):
    # precision, recall, f1, micro_precision, micro_recall, micro_f1
    return {
        "precision": tup[0],
        "recall": tup[1],
        "f1": tup[2],
        "micro_precision": tup[3],
        "micro_recall": tup[4],
        "micro_f1": tup[5],
    }


# ground_truth_test = {"test": test_article}
eval_functions = {
    "NER": eval_submission_NER,
    "NERD": eval_submission_NERD,
    "mention_level_RE": eval_submission_mention_level_RE,
    "concept_level_RE": eval_submission_concept_level_RE,
}
scores_list = []
for typ, eval_fun in eval_functions.items():
    print(f"Evaluating {typ}...")
    scores = eval_fun(
        prepare_for_eval(annotated_articles), prepare_for_eval(dev_articles_reduced)
    )
    scores=tuple_to_scores(scores)
    scores["type"] = typ
    scores_list.append(scores)

import pandas as pd
df_scores = pd.DataFrame(scores_list)
df_scores
